<img src="https://github.com/hernancontigiani/ceia_memorias_especializacion/raw/master/Figures/logoFIUBA.jpg" width="500" align="center">


# Procesamiento de lenguaje natural
## Custom embedddings con Gensim



### Objetivo
El objetivo es utilizar documentos / corpus para crear embeddings de palabras basado en ese contexto. Se utilizará canciones de bandas para generar los embeddings, es decir, que los vectores tendrán la forma en función de como esa banda haya utilizado las palabras en sus canciones.

In [188]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import multiprocessing
try:
  from gensim.models import Word2Vec
except:
  !pip install gensim
  from gensim.models import Word2Vec

### Datos
Utilizaremos como dataset canciones de bandas de habla inglesa.

In [189]:
# Descargar la carpeta de dataset
import os
import platform
if os.access('./songs_dataset', os.F_OK) is False:
    if os.access('songs_dataset.zip', os.F_OK) is False:
        if platform.system() == 'Windows':
            !curl https://raw.githubusercontent.com/FIUBA-Posgrado-Inteligencia-Artificial/procesamiento_lenguaje_natural/main/datasets/songs_dataset.zip -o songs_dataset.zip
        else:
            !wget songs_dataset.zip https://github.com/FIUBA-Posgrado-Inteligencia-Artificial/procesamiento_lenguaje_natural/raw/main/datasets/songs_dataset.zip
    !unzip -q songs_dataset.zip
else:
    print("El dataset ya se encuentra descargado")

El dataset ya se encuentra descargado


In [190]:
# Posibles bandas
os.listdir("./songs_dataset/")

['adele.txt',
 'al-green.txt',
 'alicia-keys.txt',
 'amy-winehouse.txt',
 'beatles.txt',
 'bieber.txt',
 'bjork.txt',
 'blink-182.txt',
 'bob-dylan.txt',
 'bob-marley.txt',
 'britney-spears.txt',
 'bruce-springsteen.txt',
 'bruno-mars.txt',
 'cake.txt',
 'dickinson.txt',
 'disney.txt',
 'dj-khaled.txt',
 'dolly-parton.txt',
 'dr-seuss.txt',
 'drake.txt',
 'eminem.txt',
 'janisjoplin.txt',
 'jimi-hendrix.txt',
 'johnny-cash.txt',
 'joni-mitchell.txt',
 'kanye-west.txt',
 'kanye.txt',
 'Kanye_West.txt',
 'lady-gaga.txt',
 'leonard-cohen.txt',
 'lil-wayne.txt',
 'Lil_Wayne.txt',
 'lin-manuel-miranda.txt',
 'lorde.txt',
 'ludacris.txt',
 'michael-jackson.txt',
 'missy-elliott.txt',
 'nickelback.txt',
 'nicki-minaj.txt',
 'nirvana.txt',
 'notorious-big.txt',
 'notorious_big.txt',
 'nursery_rhymes.txt',
 'patti-smith.txt',
 'paul-simon.txt',
 'prince.txt',
 'r-kelly.txt',
 'radiohead.txt',
 'rihanna.txt']

In [191]:
# Armar el dataset utilizando salto de línea para separar las oraciones/docs
df = pd.read_csv('songs_dataset/radiohead.txt', sep='/n', header=None)
df.head()

C:\Users\agust\AppData\Local\Temp\ipykernel_48640\2728444896.py:2: ParserWarning: Falling back to the 'python' engine because the 'c' engine does not support regex separators (separators > 1 char and different from '\s+' are interpreted as regex); you can avoid this warning by specifying engine='python'.
  df = pd.read_csv('songs_dataset/radiohead.txt', sep='/n', header=None)


,0
0,"Come on, come on"
1,You think you drive me crazy
2,"Come on, come on"
3,You and whose army?
4,You and your cronies


In [192]:
print("Cantidad de documentos:", df.shape[0])

Cantidad de documentos: 2343


### 1 - Preprocesamiento

In [193]:
from tensorflow.keras.preprocessing.text import text_to_word_sequence

sentence_tokens = []
# Recorrer todas las filas y transformar las oraciones
# en una secuencia de palabras (esto podría realizarse con NLTK o spaCy también)
for _, row in df[:None].iterrows():
    sentence_tokens.append(text_to_word_sequence(row[0]))

In [194]:
# Demos un vistazo
sentence_tokens[:2]

[['come', 'on', 'come', 'on'], ['you', 'think', 'you', 'drive', 'me', 'crazy']]

### 2 - Crear los vectores (word2vec)

In [195]:
from gensim.models.callbacks import CallbackAny2Vec
# Durante el entrenamiento gensim por defecto no informa el "loss" en cada época
# Sobrecargamos el callback para poder tener esta información
class callback(CallbackAny2Vec):
    """
    Callback to print loss after each epoch
    """
    def __init__(self):
        self.epoch = 0

    def on_epoch_end(self, model):
        loss = model.get_latest_training_loss()
        if self.epoch == 0:
            print('Loss after epoch {}: {}'.format(self.epoch, loss))
        else:
            print('Loss after epoch {}: {}'.format(self.epoch, loss- self.loss_previous_step))
        self.epoch += 1
        self.loss_previous_step = loss

In [196]:
# Crearmos el modelo generador de vectores
# En este caso utilizaremos la estructura modelo Skipgram
w2v_model = Word2Vec(min_count=5,    # frecuencia mínima de palabra para incluirla en el vocabulario
                     window=2,       # cant de palabras antes y desp de la predicha
                     vector_size=300,       # dimensionalidad de los vectores
                     negative=20,    # cantidad de negative samples... 0 es no se usa
                     workers=1,      # si tienen más cores pueden cambiar este valor
                     sg=1)           # modelo 0:CBOW  1:skipgram

In [197]:
# Obtener el vocabulario con los tokens
w2v_model.build_vocab(sentence_tokens)

In [198]:
# Cantidad de filas/docs encontradas en el corpus
print("Cantidad de docs en el corpus:", w2v_model.corpus_count)

Cantidad de docs en el corpus: 2343


In [199]:
# Cantidad de words encontradas en el corpus
print("Cantidad de words distintas en el corpus:", len(w2v_model.wv.index_to_key))

Cantidad de words distintas en el corpus: 383


### 3 - Entrenar embeddings

In [200]:
# Entrenamos el modelo generador de vectores
# Utilizamos nuestro callback
w2v_model.train(sentence_tokens,
                 total_examples=w2v_model.corpus_count,
                 epochs=20,
                 compute_loss = True,
                 callbacks=[callback()]
                 )

Loss after epoch 0: 88968.21875
Loss after epoch 1: 40574.34375
Loss after epoch 2: 38721.125
Loss after epoch 3: 37832.328125
Loss after epoch 4: 37930.6875
Loss after epoch 5: 37624.265625
Loss after epoch 6: 37308.59375
Loss after epoch 7: 37561.71875
Loss after epoch 8: 36464.90625
Loss after epoch 9: 36703.34375
Loss after epoch 10: 37012.0625
Loss after epoch 11: 36185.75
Loss after epoch 12: 35439.21875
Loss after epoch 13: 35838.8125
Loss after epoch 14: 35036.4375
Loss after epoch 15: 35626.8125
Loss after epoch 16: 34416.625
Loss after epoch 17: 34284.875
Loss after epoch 18: 34361.4375
Loss after epoch 19: 33634.875


(117188, 236060)

La loss baja bastante entre la época 0 (~89k) y la 19 (~34k), lo que indica que el modelo fue aprendiendo. En el medio hay algunas épocas donde sube un poco, pero en general la tendencia es hacia abajo. Dado que el corpus es relativamente chico (2343 líneas, 383 palabras distintas con frecuencia mínima 5), tampoco se espera una convergencia perfectamente suave.

### 4 - Ensayar

In [201]:
# Palabras que MÁS se relacionan con...:
w2v_model.wv.most_similar(positive=["creep"], topn=10)

[('weirdo', 0.9951853156089783),
 ('bunker', 0.981267511844635),
 ('such', 0.981124758720398),
 ("who's", 0.981106162071228),
 ('height', 0.9806836247444153),
 ('world', 0.9806110262870789),
 ('hole', 0.9805196523666382),
 ('stuffed', 0.9802872538566589),
 ('wall', 0.97955721616745),
 ('trapped', 0.9792452454566956)]

Tiene mucho sentido que *weirdo* sea la palabra más cercana a *creep*, porque en la canción aparecen juntas en la misma línea ("I'm a creep, I'm a weirdo"). El resto de los vecinos — *hole, trapped, wall, bunker* — también tienen lógica: en las letras de Radiohead hay mucho vocabulario de encierro y de sentirse fuera de lugar. *Creep* es quizás la canción más representativa de esa sensación.

In [202]:
# Palabras que MENOS se relacionan con...:
w2v_model.wv.most_similar(negative=["love"], topn=10)

[('uptight', -0.04739701375365257),
 ('oh', -0.6002485752105713),
 ("there'll", -0.602763295173645),
 ('more', -0.6061744689941406),
 ('ym', -0.6084063053131104),
 ('fo', -0.6115169525146484),
 ('lies', -0.6140448451042175),
 ('no', -0.6217774748802185),
 ('ah', -0.6221923232078552),
 ('eat', -0.634705662727356)]

Acá aparecen *ym* y *fo* que son artefactos del texto (palabras mal tokenizadas o al revés), así que no tienen valor semántico real. Ignorando eso, llama la atención que *lies* y *no* aparezcan como los más lejanos a *love*, en lugar de algo como *hate*. En Radiohead la idea de "lo opuesto al amor" no pasa por el odio sino más bien por la mentira y la negación. *Uptight* en cambio tiene similitud casi nula (−0.047), lo que significa que directamente no tiene relación con *love* en este corpus.

In [203]:
# Palabras que MÁS se relacionan con...:
w2v_model.wv.most_similar(positive=["everything"], topn=10)

[('this', 0.9730702042579651),
 ('walls', 0.9723333120346069),
 ('happening', 0.9712907075881958),
 ('birds', 0.9695520401000977),
 ('cut', 0.9672297239303589),
 ('panic', 0.9661234617233276),
 ('earth', 0.9660895466804504),
 ('wall', 0.9658069014549255),
 ('killing', 0.9653630256652832),
 ("he's", 0.9649078249931335)]

Lo interesante de *everything* es que sus vecinos no son palabras tranquilas o positivas, sino *panic, killing, walls, happening, cut*. En Radiohead el "todo" aparece en contextos de caos, no de plenitud. *Birds* y *earth* también están ahí, lo que refuerza un poco la idea ambiental/apocalíptica que aparece en canciones como *Idioteque*. En resumen, en este corpus *everything* está más cerca del colapso que de la completitud.

In [204]:
# Palabras que MÁS se relacionan con...:
w2v_model.wv.most_similar(positive=["weird"], topn=5)

[('words', 0.9991586804389954),
 ('burn', 0.9987322092056274),
 ('eye', 0.9987316727638245),
 ('fall', 0.9987066388130188),
 ('people', 0.9985865354537964)]

Me llama la atención la presencia de *eye* y *people* cerca de *weird*: pareciera que en las letras la rareza siempre está ligada a la mirada del otro, a ser observado. *Burn* y *fall* le suman un peso bastante negativo, como si ser raro implicara también una especie de castigo o caída. No es un adjetivo que aparezca en contextos neutros.

In [205]:
# Ensayar con una palabra que no está en el vocabulario:
try:
    w2v_model.wv.most_similar(negative=["xyzfakeword"])
except KeyError as e:
    print(f"La palabra no se encuentra en el vocabulario: {e}")

La palabra no se encuentra en el vocabulario: "Key 'xyzfakeword' not present in vocabulary"


In [206]:
# el método `get_vector` permite obtener los vectores:
vector_love = w2v_model.wv.get_vector("love")
print(vector_love)

[ 3.37626436e-03  4.80917357e-02 -7.14344382e-02  7.61521310e-02
 -1.79719497e-02 -1.78191870e-01  1.06714688e-01  3.83953273e-01
  8.23621303e-02 -1.66605171e-02  3.26546058e-02  6.14719875e-02
  3.16183642e-02 -1.87131446e-02 -2.60132812e-02 -3.65004577e-02
  1.32146567e-01 -4.66845408e-02  4.12178114e-02 -8.73279050e-02
 -7.71199912e-02  1.45091936e-01  1.07295647e-01 -3.32668126e-02
  8.78842622e-02  1.20259477e-02  3.00502181e-02 -6.10791966e-02
 -3.09657212e-02 -8.79980624e-03 -7.26322010e-02  3.10212374e-02
  6.06012419e-02 -2.95871161e-02 -6.98665529e-02  1.54989019e-01
  2.90349163e-02 -1.41184837e-01  1.28462659e-02  7.53032267e-02
 -1.06770694e-01  2.73098946e-02  1.14681505e-01 -6.76414892e-02
  1.37863055e-01  2.06131637e-01  2.54539168e-03  1.57235675e-02
 -9.71881859e-03  1.04390472e-01  1.02728046e-01  3.03381449e-03
 -4.16624397e-02 -9.26782861e-02  5.62940016e-02  6.52501881e-02
  1.17596127e-01 -4.58303727e-02 -6.13029767e-03  1.53623506e-01
 -9.16631743e-02  1.51461

In [207]:
# el método `most_similar` también permite comparar a partir de vectores
w2v_model.wv.most_similar(vector_love)

[('love', 1.0),
 ('ever', 0.9823560118675232),
 ('had', 0.9809578657150269),
 ('their', 0.9804283380508423),
 ('did', 0.9796867966651917),
 ('laugh', 0.978361189365387),
 ('tried', 0.9780083894729614),
 ('choke', 0.9777578711509705),
 ('always', 0.9773685932159424),
 ("what's", 0.9772995114326477)]

In [208]:
# Palabras que MÁS se relacionan con...:
w2v_model.wv.most_similar(positive=["love"], topn=10)

[('ever', 0.9823560118675232),
 ('had', 0.9809578657150269),
 ('their', 0.9804283380508423),
 ('did', 0.9796867966651917),
 ('laugh', 0.978361189365387),
 ('tried', 0.9780084490776062),
 ('choke', 0.9777578711509705),
 ('always', 0.9773685932159424),
 ("what's", 0.9772995114326477),
 ('tonight', 0.9770829081535339)]

Buscar por el vector directamente da prácticamente el mismo resultado que buscar por la palabra, lo cual tiene sentido porque es la misma representación.

Lo que más me sorprende de los vecinos de *love* es la cantidad de verbos en pasado: *had, did, tried*. En Radiohead el amor no parece algo del presente, es algo que ya pasó o que se intentó y no funcionó del todo. *Choke* también es bastante fuerte para estar en ese grupo. No hay nada parecido a *kiss* o *hold*, que serían los vecinos típicos en un corpus pop más convencional.

### 5 - Visualizar agrupación de vectores

In [209]:
from sklearn.decomposition import IncrementalPCA
from sklearn.manifold import TSNE
import numpy as np

def reduce_dimensions(model, num_dimensions = 2 ):

    vectors = np.asarray(model.wv.vectors)
    labels = np.asarray(model.wv.index_to_key)

    tsne = TSNE(n_components=num_dimensions, random_state=0)
    vectors = tsne.fit_transform(vectors)

    return vectors, labels

In [210]:
# Graficar los embedddings en 2D
import plotly.graph_objects as go
import plotly.express as px

vecs, labels = reduce_dimensions(w2v_model)

MAX_WORDS=200
fig = px.scatter(x=vecs[:MAX_WORDS,0], y=vecs[:MAX_WORDS,1], text=labels[:MAX_WORDS])
fig.show(renderer="colab") # esto para plotly en colab

En el gráfico se pueden distinguir algunos grupos. Las palabras funcionales (*the, a, and, to, of*) aparecen bastante juntas, lo que tiene sentido porque aparecen en casi todos los contextos posibles y el modelo no puede diferenciarlas mucho entre sí.

Más interesante es ver que *trapped, hole, wall, bunker, creep, weirdo* quedan relativamente cerca, que es exactamente el vocabulario de encierro y marginalidad que ya venía apareciendo en los experimentos anteriores.

En otra zona del gráfico aparecen palabras como *panic, killing, earth, birds, everything*, que tienen más que ver con la dimensión colectiva y catastrófica de algunas canciones.

Algo que hay que tener en cuenta es que las similitudes en general son muy altas (muchos pares por encima de 0.97), probablemente porque el vocabulario es chico. Con más datos los clusters estarían más separados y sería más fácil distinguirlos visualmente.

In [211]:
# Graficar los embedddings en 3D

vecs, labels = reduce_dimensions(w2v_model,3)

fig = px.scatter_3d(x=vecs[:MAX_WORDS,0], y=vecs[:MAX_WORDS,1], z=vecs[:MAX_WORDS,2],text=labels[:MAX_WORDS])
fig.update_traces(marker_size = 2)
fig.show(renderer="colab") # esto para plotly en colab

In [212]:
# También se pueden guardar los vectores y labels como tsv para graficar en
# http://projector.tensorflow.org/


vectors = np.asarray(w2v_model.wv.vectors)
labels = list(w2v_model.wv.index_to_key)

np.savetxt("vectors.tsv", vectors, delimiter="\t")

with open("labels.tsv", "w") as fp:
    for item in labels:
        fp.write("%s\n" % item)

### Consigna del desafío 2

**Cada experimento realizado debe estar acompañado de una explicación o interpretación de lo observado**

Recuerden que su notebook de entrega debe poder correrse de inicio a fin sin la aparición de errores.

- Crear sus propios vectores con Gensim basado en lo visto en clase con un corpus propio (revisar enlaces sugeridos en clase 2 sobre opciones de dataset)
- Elegir términos de interés y buscar términos más similares y menos similares.
- Realizar una reduccion de dimensionalidad a los embeddings, llevándolos a 2 dimensiones. Graficar los embeddings proyectados y seleccionar una cantidad de términos (variable MAX_WORDS) de forma tal que la visualización sea adecuada.
- Inspeccionar el grafico y buscar pequeños grupos de palabras que puedan formarse. Interpretarlos e intentar obtener conclusiones. En lo posible, acompañar los grupos de palabras con capturas (y pegarlas en celdas de texto)